### Step 1: Environment Setup

**Description:** Install the necessary libraries for fine-tuning the Ministral 3 model, including Hugging Face `transformers`, `trl` for supervised fine-tuning, and `bitsandbytes` for memory optimization.

In [ ]:
%%capture
!pip install -U transformers==5.3.0 trl peft datasets accelerate bitsandbytes mistral_common liger-kernel flash-attn

### Step 2: Hardware and CUDA Verification

**Description:** Check the environment to ensure the GPU is recognized correctly and verify the available VRAM to determine if the setup can handle the model.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_props = torch.cuda.get_device_properties(0)
    print(f"GPU: {device_props.name}")
    print(f"Total VRAM: {device_props.total_memory / 1e9:.2f} GB")


### Step 3: Library Imports

**Description:** Load core modules for model architecture, dataset management, and the SFT (Supervised Fine-Tuning) trainer.

In [ ]:
import torch
import gc
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    Mistral3ForConditionalGeneration, 
    MistralCommonBackend
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, TaskType

### Step 4: Model and Tokenizer Initialization

**Description:** Load the model with `bfloat16` precision and configure the chat template. This step also includes freezing the vision-related parameters (vision tower and projector) to focus training strictly on the language capabilities.

In [ ]:
model_id = "0xA50C1A1/Ministral-3-8B-Instruct-2512-BF16-SOM-MPOA"

# Load the model in BF16
base_model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
    trust_remote_code=True
)

# Load and configure tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Define custom chat template for Ministral 3
tokenizer.chat_template = """
{%- if messages[0]['role'] == 'system' -%}
    {{- bos_token + '[SYSTEM_PROMPT]' + messages[0]['content'] + '[/SYSTEM_PROMPT]' -}}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {{- bos_token -}}
    {%- set loop_messages = messages -%}
{%- endif -%}

{%- for message in loop_messages -%}
    {%- if message['role'] == 'user' -%}
        {{- '[INST]' + message['content'] + '[/INST]' -}}
    {%- elif message['role'] == 'assistant' -%}
        {%- generation -%}
            {{- message['content'] + eos_token -}}
        {%- endgeneration -%}
    {%- endif -%}
{%- endfor -%}

{%- if add_generation_prompt -%}
    {%- generation -%}{%- endgeneration -%}
{%- endif -%}
""".strip()

### Step 5: PEFT/LoRA Configuration

**Description:** Configure and apply LoRA to the language model components. This allows efficient fine-tuning with high rank while keeping vision components frozen.

In [ ]:
peft_config = LoraConfig(
    r=64,
    lora_alpha=64,
    target_modules="all-linear",
    exclude_modules=[
        "vision_tower",
        "multi_modal_projector"
    ],
    use_dora=True,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply PEFT to the model
model = get_peft_model(base_model, peft_config)

# Print trainable parameters
model.print_trainable_parameters()

### Step 6: Dataset Loading

**Description:** Import the training dataset from Hugging Face. We apply a shuffle to ensure better stochastic gradient descent during the training process.

In [ ]:
dataset = load_dataset("0xA50C1A1/rp_dataset", split="train").shuffle(seed=3407)
print(f"Dataset loaded: {len(dataset)} samples ready.")

### Step 7: SFT Trainer Configuration

**Description:** Set up the fine-tuning hyper-parameters. We use `SFTConfig` for efficient memory management, including gradient checkpointing and NEFTune noise for better generalization.

In [ ]:
sft_config = SFTConfig(
    output_dir="outputs",

    # Save to HF
    push_to_hub=True,
    hub_private_repo=True,
    save_only_model=True,
    hub_model_id="0xA50C1A1/Ministral-3-8B-Instruct-2512-DoRA",
    hub_strategy="end",

    dataset_text_field="messages",
    packing=True,
    padding_free=True,
    assistant_only_loss=True,
    
    # Training Hyperparameters
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    
    # Precision and Efficiency
    bf16=True,
    use_liger_kernel=True,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Optimizer & Scheduler
    optim="adamw_torch_fused",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    max_length=8192,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    
    # Logging
    logging_steps=5,
    report_to="none",
    
    # Regularization
    seed=3407,
    neftune_noise_alpha=5,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

### Step 8: Training Execution

**Description:** Initializing the training loop. We perform a manual garbage collection and clear the CUDA cache to prevent Out-Of-Memory (OOM) errors at the start of the run.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

trainer.train()

### Step 9: Model Inference and Testing

**Description:** Run a qualitative test on the fine-tuned model. We switch the model to evaluation mode and generate a response using the updated chat template to verify the learning results.

In [ ]:
# Clear memory and set to eval mode
torch.cuda.empty_cache()
model.eval()

# Sample prompt
messages = [
    {"role": "system", "content": "Roleplay as a tavern keeper. Describe your actions using *asterisks* and wrap your speech in \"double quotes\"."},
    {"role": "user", "content": "*I walk into the tavern and ask for an ale.*"}
]

# Prepare inputs
inputs = tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True, 
    return_tensors="pt",
    return_dict=True
).to(model.device)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=300, 
        do_sample=True, 
        temperature=1.0,
        min_p=0.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
    )

# Decode output (removing the input tokens)
input_length = inputs["input_ids"].shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print(f"Prompt: {messages[1]['content']}")
print(f"Response:\n{response}")

### Step 10: Model Export and Upload to Hugging Face Hub

**Description:** Authenticate with the Hugging Face Hub and upload the fine-tuned model, tokenizer, and training configuration. This allows for easy sharing, versioning, and deployment of your model.

##### 10.1: Hub Authentication

In [ ]:
from huggingface_hub import login

# Put your HF Write Token here
# It is recommended to use an environment variable or secret for security
HF_TOKEN = "your_hf_token" 

login(token=HF_TOKEN)

##### 10.2: Push to Hub

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel

adapter_path = "outputs/checkpoint-96"

base_repo = "0xA50C1A1/Ministral-3-8B-Instruct-2512-BF16-SOM-MPOA"
tokenizer_repo = "mistralai/Ministral-3-8B-Instruct-2512-BF16"
hub_model_id = "0xA50C1A1/Ministral-3-8B-Instruct-2512-DoRA"

print("Loading base model...")
base_model = AutoModelForImageTextToText.from_pretrained(
    base_repo,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

print("Merging LoRA into base model...")
model = model.merge_and_unload()
print("Merge complete!")

print("Loading original tokenizer (with original chat_template)...")
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_repo,
    trust_remote_code=True
)

save_path = "merged_model"

print("Saving locally...")
model.save_pretrained(save_path, safe_serialization=True)
tokenizer.save_pretrained(save_path)

print("Pushing to Hub...")
model.push_to_hub(hub_model_id, private=True)
tokenizer.push_to_hub(hub_model_id, private=True)

print(f"✅ https://huggingface.co/{hub_model_id}")